# Day 1 - Structure & Diagnostics - SOLUTIONS

> Instructor copy. Every TODO is filled in and every question answered.
> The student copy is the same notebook with these cells blanked.

## Setup

Everything the labs need lives in `coursekit`. If the next cell fails, run
`python scripts/check_env.py` from the repo root and fix what it reports.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from coursekit import checks
from coursekit import datasets as D
from coursekit import plotting as P

P.use_course_style()
print("ready")

---
# Exercise 1.1 - Name the pattern

*Follows segment 1. 10 minutes. No modelling - look and argue.*

Six series are plotted below. For each one decide:

- Is there a **trend**?
- Is there **seasonality** (a *fixed, known* period)?
- Is there a **cycle** (rises and falls at *no* fixed period)?
- Is the seasonal swing **additive** (constant size) or **multiplicative**
  (grows with the level)?

In [ ]:
series = {
    "spine": D.spine(),
    "beer": D.beer(),
    "lynx": D.lynx(),
    "noise": D.white_noise(n=300, seed=7),
    "souvenirs": D.souvenirs(),
    "canadian_gas": D.canadian_gas(),
}

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, (name, df) in zip(axes.ravel(), series.items()):
    P.plot_series(df, ax=ax, title=name)
P.thin_xticks(axes, n=3)
plt.show()

Fill in your calls below. Use any of: `"trend"`, `"seasonality"`, `"cycle"`,
`"none"`, `"additive"`, `"multiplicative"`.

In [ ]:
answers = {
    "spine":        ["trend", "seasonality", "multiplicative"],
    "beer":         ["trend", "seasonality", "additive"],
    "lynx":         ["cycle"],
    "noise":        ["none"],
    "souvenirs":    ["trend", "seasonality", "multiplicative"],
    "canadian_gas": ["trend", "seasonality", "multiplicative"],
}

checks.check_ex_1_1(answers)

> **Discussion.** `lynx` is the one that catches people. Its peaks are 8-11
> years apart - never the same gap twice - so it is a *cycle*, not seasonality.
> Nothing in the calendar produces it.
>
> `canadian_gas` is worth a second look too: its seasonal *shape* changes over
> the decades, not just its size. STL will show you that in exercise 1.4.

---
# Exercise 1.2 - Load, verify, and look

*Follows segment 2. 15 minutes.*

The spine for this whole course is Victorian takeaway-food turnover: monthly,
1982-2018.

**Before plotting anything, verify the timestamps.** A silent gap shifts every
seasonal lag after it, and nothing downstream will warn you.

In [ ]:
spine = D.spine()
spine.head()

In [ ]:
inferred_freq = pd.infer_freq(spine["ds"])
n_duplicates  = int(spine["ds"].duplicated().sum())
n_expected    = len(pd.date_range(spine["ds"].min(), spine["ds"].max(), freq="MS"))
n_actual      = len(spine)

print(f"inferred frequency : {inferred_freq}")
print(f"duplicate stamps   : {n_duplicates}")
print(f"expected / actual  : {n_expected} / {n_actual}")

checks.check_ex_1_2(spine)

This series is clean. Most are not - so here is what a gap actually costs.
Run this and compare the two seasonal lags.

In [ ]:
# Drop three months at random and see what happens to the seasonal structure.
rng = np.random.default_rng(0)
holes = rng.choice(np.arange(100, 300), size=3, replace=False)
broken = spine.drop(index=holes).reset_index(drop=True)

r_full, _ = P.acf_values(spine["y"], nlags=24)
r_broken, _ = P.acf_values(broken["y"], nlags=24)
print(f"r_12 with a complete calendar : {r_full[11]:.3f}")
print(f"r_12 after dropping 3 months  : {r_broken[11]:.3f}")
print("\nThe rows still line up. The CALENDAR does not.")

**Repairing a gap.** Reindex onto the full date range, then decide what the
missing values mean - interpolate, carry forward, or leave `NaN` and use a
model that tolerates them. Never let the gap stay *invisible*.

In [ ]:
full_index = pd.date_range(broken["ds"].min(), broken["ds"].max(), freq="MS")
repaired = (broken.set_index("ds")
                  .reindex(full_index)
                  .rename_axis("ds")
                  .reset_index())
repaired["unique_id"] = repaired["unique_id"].ffill()
print(f"rows: {len(broken)} -> {len(repaired)},  NaNs now visible: {repaired['y'].isna().sum()}")
repaired["y"] = repaired["y"].interpolate()
print(f"after interpolation, NaNs: {repaired['y'].isna().sum()}")

Now the three plots. Each answers a different question.

In [ ]:
sp = D.add_calendar(spine)

# 1. time plot
ax = P.plot_series(spine, title="Spine - monthly turnover")
plt.show()

# 2. seasonal plot
fig, ax = plt.subplots(figsize=(9, 4.2))
P.seasonal_plot(sp, "year", "month", ax=ax, title="One line per year")
ax.set_xticks(range(1, 13))
plt.show()

# 3. subseries plot
fig, axes = P.subseries_plot(sp, "month", title="One panel per month")
plt.show()

**Write your answer here.** In two or three sentences: what is going on in this
series? Mention the trend, the seasonal shape, whether the swing is growing, and
anything unusual around 2009.


*Your answer:*


*Answer.* Turnover rises roughly eight-fold from 1982 to 2018, with a clear
December peak and February trough every year. The seasonal swing grows with the
level, so the series is multiplicative - that is what the Box-Cox transform in
1.4 will fix. Growth flattens noticeably around 2009-2010 (the financial
crisis), which is a level effect rather than a seasonal one; the seasonal plot
shows the *shape* stays put while the level stalls.

### Stretch

Pick a second retail series with `D.retail_all()` and contrast it with the
spine. Is its seasonal shape the same? Does it peak in December too?

In [ ]:
allr = D.retail_all()
other = allr[allr["unique_id"] == "New South Wales / Newspaper and book retailing"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
P.plot_series(spine, ax=axes[0], title="Spine - takeaway food")
P.plot_series(other, ax=axes[1], title="Newspapers and books", color=P.ORANGE)
P.thin_xticks(axes, n=4)
plt.show()
print("Both peak in December, but the book series declines after 2005 while "
      "takeaway keeps growing - opposite trends, same seasonal calendar.")

---
# Exercise 1.3 - Read the correlogram

*Follows segment 3. 15 minutes.*

First, the matching game. Four correlograms below, in a scrambled order.
Match each to its series.

In [ ]:
mystery = {
    "A": D.white_noise(n=400, seed=11),
    "B": D.spine(),
    "C": D.beer(),
    "D": D.lynx(),
}
order = ["C", "A", "D", "B"]      # the plots are drawn in THIS order

fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, key in zip(axes, order):
    P.acf_plot(mystery[key]["y"], nlags=30, ax=ax, title=f"correlogram {order.index(key) + 1}")
plt.show()

Which correlogram belongs to which series? Say *why* - name the feature you
used (slow decay, spikes at a period, everything inside the bounds).


*Your answer:*


*Answer.*

1. **beer** - spikes at lags 4, 8, 12 (quarterly data, m = 4) with little decay.
2. **white noise** - every spike inside the bounds.
3. **lynx** - a slow *wave*: positive at short lags, negative around lag 5,
   positive again near lag 10. A cycle shows as an oscillating ACF, not as
   spikes at a fixed multiple.
4. **spine** - slow decay from near 1.0, the signature of a strong trend, with
   a seasonal ripple riding on top.

In [ ]:
noise = D.white_noise(n=len(spine), seed=7)

acf_spine, bound = P.acf_values(spine["y"], nlags=36)
acf_noise, _ = P.acf_values(noise["y"], nlags=36)

print(f"bound = {bound:.4f}")
print(f"spine  r_1 = {acf_spine[0]:.3f}   r_12 = {acf_spine[11]:.3f}")
print(f"noise: fraction of lags outside the bounds = "
      f"{(np.abs(acf_noise) > bound).mean():.1%}")

checks.check_ex_1_3(acf_spine, acf_noise, bound)

**Question.** Roughly 5% of white-noise autocorrelations land outside the
bounds *by construction*. If you plot 36 lags, how many spikes outside the band
should stop worrying you?


*Your answer:*


*Answer.* About 36 x 0.05 = 1.8, so one or two stray spikes are exactly what
white noise looks like. Treat the bounds as a null hypothesis about a *single*
lag, not as a per-plot decision rule. What matters is a pattern - a run of
spikes, or a spike at a meaningful lag like 12.

---
# Exercise 1.4 - Transform and decompose

*Follows segment 4. 15 minutes.*

The spine's seasonal swing grows with its level. Stabilise it first, then split
it into trend, seasonal and remainder.

In [ ]:
from coreforecast.scalers import boxcox, boxcox_lambda
from statsmodels.tsa.seasonal import STL

lam = boxcox_lambda(spine["y"].to_numpy(), method="loglik")
yt = boxcox(spine["y"].to_numpy(), lam)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
P.plot_series(spine, ax=axes[0], title="original")
axes[1].plot(spine["ds"], yt, color=P.ORANGE, lw=0.9)
axes[1].set_title(f"Box-Cox, lambda = {lam:.3f}")
plt.show()

In [ ]:
res = STL(yt, period=12, robust=True).fit()
dcmp = spine.assign(
    transformed=yt,
    trend=np.asarray(res.trend),
    seasonal=np.asarray(res.seasonal),
    remainder=np.asarray(res.resid),
)

fig, axes = P.decomposition_plot(
    dcmp, ["transformed", "trend", "seasonal", "remainder"], "STL")
plt.show()

checks.check_ex_1_4(dcmp, lam)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4))
ax.plot(spine["ds"], dcmp["transformed"], color=P.GREY, lw=0.8, label="observed")
ax.plot(spine["ds"], dcmp["transformed"] - dcmp["seasonal"], color=P.ORANGE,
        lw=1.2, label="seasonally adjusted")
ax.set(title="Seasonally adjusted")
ax.legend(frameon=False)
plt.show()

### Stretch

Run a **classical** decomposition (`seasonal_decompose`) on the same series and
compare it with STL. Look hard at the first and last six months.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

cl = seasonal_decompose(yt, period=12, model="additive")
fig, axes = plt.subplots(2, 1, figsize=(9.5, 5), sharex=True)
axes[0].plot(spine["ds"], np.asarray(res.trend), color=P.BLUE, lw=1.2)
axes[0].set_title("STL trend - defined everywhere")
axes[1].plot(spine["ds"], np.asarray(cl.trend), color=P.ORANGE, lw=1.2)
axes[1].set_title("Classical trend - six months missing at each end")
plt.show()

print("Classical decomposition uses a centred moving average, so it cannot "
      "estimate the trend for the first and last m/2 observations - exactly "
      "the end you care about when forecasting. It also forces ONE seasonal "
      "shape for all 37 years; STL lets the shape evolve.")

---
# Exercise 1.5 - Features across a portfolio

*Follows segment 5. 16 minutes.*

One series is a plot. A hundred and forty-eight series need **numbers**.

Compute strength of trend and strength of seasonality for every Australian
retail series, then use them to find the interesting ones.

In [ ]:
def stl_features(g):
    """Return trend and seasonal strength for one series (long format)."""
    y = np.log(np.clip(g["y"].to_numpy(), 1e-6, None))
    r = STL(y, period=12, robust=True).fit()
    R, T_, S = np.asarray(r.resid), np.asarray(r.trend), np.asarray(r.seasonal)
    var_r = np.var(R)
    trend_strength = max(0.0, 1 - var_r / np.var(T_ + R))
    seasonal_strength = max(0.0, 1 - var_r / np.var(S + R))
    return pd.Series({"trend_strength": trend_strength,
                      "seasonal_strength": seasonal_strength})


allr = D.retail_all()
feat = (allr.groupby("unique_id")[["y"]]
            .apply(stl_features, include_groups=False)
            .reset_index())

checks.check_ex_1_5(feat)
feat.head()

In [ ]:
print("MOST seasonal")
print(feat.nlargest(5, "seasonal_strength").to_string(index=False))
print("\nLEAST seasonal")
print(feat.nsmallest(5, "seasonal_strength").to_string(index=False))

**Now check the numbers meant what you think.** Plot the most and the least
seasonal series side by side. If the feature is doing its job, the difference
should be obvious to the eye.

In [ ]:
most = feat.nlargest(1, "seasonal_strength")["unique_id"].iloc[0]
least = feat.nsmallest(1, "seasonal_strength")["unique_id"].iloc[0]

fig, axes = plt.subplots(2, 1, figsize=(10, 5.5))
for ax, uid, col in [(axes[0], most, P.ORANGE), (axes[1], least, P.BLUE)]:
    g = allr[allr["unique_id"] == uid]
    P.plot_series(g, ax=ax, color=col,
                  title=f"{uid}  (F_S = {feat.loc[feat['unique_id'] == uid, 'seasonal_strength'].iloc[0]:.2f})")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.scatter(feat["trend_strength"], feat["seasonal_strength"], s=26,
           color=P.BLUE, alpha=0.6)
mine = feat[feat["unique_id"] == D.SPINE_ID].iloc[0]
ax.scatter([mine["trend_strength"]], [mine["seasonal_strength"]], s=120,
           color=P.GREEN, zorder=3, label="our spine")
ax.set(xlabel="strength of trend", ylabel="strength of seasonality",
       title="148 retail series in feature space")
ax.legend(frameon=False)
plt.show()

**Question.** Trend strength is above 0.96 for nearly every series. Is that
feature useless here?


*Your answer:*


*Answer.* Useless for *discriminating between these series* - yes. But it is
still a finding: it says "everything in Australian retail trends", which tells
you that any model you pick must handle a trend, and that seasonality is the
axis worth routing on. A feature that does not vary across your portfolio is a
feature you can stop computing - after you have looked at it once.

---
## End of Day 1

You can now diagnose a series: see its patterns, measure them with the ACF,
split it into components, and summarise a whole portfolio.

Tomorrow you forecast - and, more importantly, learn how to tell whether the
forecast was any good.